# Full Process Lens For This Project

Use this section every time you start a project so you practice the same end-to-end workflow, not just isolated coding tasks.

## 1) Data Transformation and Processing (What and Why)

Raw data is rarely model-ready. Your first job is to transform data into a reliable, learnable format.

What to do in every project:
- Identify data types and expected schema.
- Handle missing values, duplicates, and inconsistent formats.
- Convert features into model-usable representations (encoding, scaling, tokenization, chunking, etc.).
- Keep transformations reproducible so train and inference use the same logic.
- Document assumptions and risks introduced by preprocessing choices.

Why this matters:
- Better preprocessing usually improves results more than switching algorithms.
- Poor preprocessing creates hidden errors that look like model failure.

## 2) Evaluating and Improving Models (What and Why)

Evaluation is not the last step. It is the loop that drives improvement.

What to do in every project:
- Start with a baseline and compare against it.
- Choose metrics tied to the real goal (not just convenience metrics).
- Inspect errors by slice (segments, classes, edge cases).
- Tune thresholds, features, prompts, retrieval settings, or hyperparameters based on evidence.
- Re-evaluate after each change and keep track of what improved and what regressed.

Why this matters:
- A model can appear good overall but fail on important cases.
- Iterative evaluation is how projects become production-ready, not just demo-ready.

## 3) Project Reflection Checklist

Before marking this project complete, confirm:
- I can explain how data was transformed and why.
- I can explain which metrics I chose and why.
- I can show at least one improvement cycle from evaluation findings.
- I can describe current limitations and next improvements.

In [1]:
# First run check (beginner safe)
import sys

print('Python version:', sys.version.split()[0])
print('Environment check passed.')
print('If later imports fail, install notebook dependencies from setup guide.')

Python version: 3.9.6
Environment check passed.
If later imports fail, install notebook dependencies from setup guide.


# Beginner Start Here (Weeks 9-10: RAG Bot)

If this is your first LLM project, this notebook is designed to guide you step by step.

## What is RAG?
RAG means Retrieval-Augmented Generation.
- Retrieval: find relevant document chunks.
- Generation: LLM writes answer using those chunks.

## Modules used in this notebook
- `langchain`: orchestration for retrieval and generation pipelines.
- `faiss`/vector store client: fast similarity search over embeddings.
- `sentence-transformers` or API embeddings: convert text into vectors.
- `rank-bm25`: keyword search for hybrid retrieval.

## Key terms
- Embedding: numeric vector representation of text.
- Chunking: splitting long documents into smaller parts.
- Similarity search: find vectors close to query vector.
- Reranking: reorder retrieved results using stronger model.
- Hallucination: confident but unsupported answer.

## How to work through this notebook
1. Build simple retrieval first.
2. Add hybrid retrieval (keyword + vector).
3. Add reranking.
4. Measure quality with eval metrics before calling it done.

# Weeks 9-10: Retrieval-Augmented Generation (RAG) Bot

## Overview
Build a chatbot that retrieves documents and generates answers using LLMs.

**Learning Goals:**
- LLM APIs (OpenAI/Hugging Face)
- Document embeddings and retrieval
- Vector databases
- Prompt engineering
- Chat interfaces

**Time:** 10-15 hours across 2 weeks

**Difficulty:** 🟡🟡 Combine LLMs + ML concepts

---

## Challenge Summary
12 challenges to build an intelligent document-aware chatbot from scratch.

## Challenge 1: Setup LLM Environment

**Goal:** Configure LLM API access and dependencies.

**Steps:**
1. Install required packages:
   - pip install openai langchain python-dotenv
   - pip install chromadb faiss-cpu  (vector databases)
2. Create .env file with API key:
   - Get key from OpenAI (or use free models)
3. Test connection to LLM
4. Verify imports work

**Expected:**
- No import errors
- API accessible
- Ready for LLM calls

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=openai_api_key)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Say hello world"},
    ]
)

print(response.choices[0].message.content)

Hello, world!


## Challenge 2: Load and Split Documents

**Goal:** Prepare documents for embedding.

**Steps:**
1. Load sample documents (from files, URLs, or create synthetic)
2. Split into chunks (512-1000 tokens each)
3. Each chunk should be meaningful (don't split mid-sentence)
4. Store chunks with metadata (document name, page number)
5. Display sample chunks

**Expected:**
- 10-50 chunks total
- Each chunk: reasonable size
- Metadata preserved

In [3]:
import wikipedia
from langchain.text_splitter import RecursiveCharacterTextSplitter


results = wikipedia.search("Barack Obama")
print("Search results:", results)


/Users/jjcatulle/Desktop/ML-AI-learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Search results: ['Barack Obama', 'Family of Barack Obama', 'Barack Obama Sr.', 'Barack Obama Presidential Center', 'Presidency of Barack Obama', 'Barack Obama Plaza', 'Michelle Obama', 'Cabinet of Barack Obama', 'Bibliography of Barack Obama', 'Barack Obama citizenship conspiracy theories']


##### fetching the document

In [4]:
title = results[0]  # Take the first search result
page=wikipedia.page(title, auto_suggest=False)
doc_text = page.content

metadata = {
    "source": "wikipedia",
    "title": page.title,
    "url": page.url
}

print("Document length:", len(doc_text))
print(doc_text[:500])

Document length: 86568
Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.
Born in Honolulu, Obama graduated from Columbia University in 1983 with a Bachelor of Arts degree in political science and later worked as a c


##### chunking the document

In [5]:


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_text(doc_text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Number of chunks: 189

First chunk:

Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.


In [6]:
chunked_docs = []

for i, chunk in enumerate(chunks):
    chunked_docs.append({
        "chunk_id": i,
        "text": chunk,
        "metadata": metadata
    })

    if i < 3:
        print(f"\nChunk {i}:\n",chunked_docs[-1])

# print(chunked_docs[0])
# print(chunked_docs[1])
# print(chunked_docs[2])
# print(chunked_docs[3])


Chunk 0:
 {'chunk_id': 0, 'text': 'Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.', 'metadata': {'source': 'wikipedia', 'title': 'Barack Obama', 'url': 'https://en.wikipedia.org/wiki/Barack_Obama'}}

Chunk 1:
 {'chunk_id': 1, 'text': 'Born in Honolulu, Obama graduated from Columbia University in 1983 with a Bachelor of Arts degree in political science and later worked as a community organizer in Chicago. In 1988, Obama enrolled in Harvard Law School, where he was the first black president of the Harvard Law Review. He became a civil rights attorney and an academic, teaching constitutional law at the University of Chicago Law School from 1992 to 2004. In 1996, Obama was elected to

## Challenge 3: Create Embeddings

**Goal:** Convert text chunks to vector embeddings.

**Steps:**
1. Use embedding model (OpenAI embeddings or sentence-transformers)
2. Generate embeddings for all chunks
3. Each embedding: 768-1536 dimensional vector
4. Verify embedding dimensions
5. Show similarity between 2 chunks (cosine similarity)

**Expected:**
- All chunks have embeddings
- Embeddings are numerical vectors
- Similar chunks have high similarity scores

##### embeding/encpoding the plain text 

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for each chunk
texts= [doc['text'] for doc in chunked_docs]

embeddings = embedding_model.encode(texts, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding shape: (189, 384)


#### test without vector DB

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

query = "Who is Barack Obama?"

query_embedding = embedding_model.encode([query])

similarities = cosine_similarity(query_embedding, embeddings)

# get best match
best_idx = similarities.argmax()

print("Best chunk:\n")
print(chunked_docs[best_idx]["text"])

Best chunk:

Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.


## Challenge 4: Store in Vector Database

**Goal:** Index embeddings for fast retrieval.

**Steps:**
1. Create vector database (Chroma or FAISS)
2. Add all document chunks with embeddings
3. Include metadata (document source, chunk index)
4. Test retrieval:
   - Query: "Tell me about machine learning"
   - Get top 3 most similar chunks
5. Verify retrieval quality

**Expected:**
- Database stores all chunks
- Retrieval returns relevant documents
- Similarity scores make sense

#### save to vector db

In [9]:
import faiss
import numpy as np

# Convert embeddings to numpy float32
embedding_matrix = np.array(embeddings).astype("float32")

# Get embedding dimension
dimension = embedding_matrix.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embedding_matrix)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 189


#### retrieval from the vector DB

In [10]:
query = "Tell me about Barack Obama"
query_embedding = embedding_model.encode([query]).astype("float32")

k = 3
distances, indices = index.search(query_embedding, k)

print("Top results:\n")

for rank, idx in enumerate(indices[0]):
    print(f"Result {rank+1}")
    print("Distance:", distances[0][rank])
    print("Chunk ID:", chunked_docs[idx]["chunk_id"])
    print("Text:", chunked_docs[idx]["text"])
    print("-" * 80)

Top results:

Result 1
Distance: 0.4816606
Chunk ID: 0
Text: Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.
--------------------------------------------------------------------------------
Result 2
Distance: 0.83714235
Chunk ID: 10
Text: African man who lived in the Colony of Virginia during the seventeenth century. Obama has described the ancestors of his grandparents as Scotch-Irish mostly. Obama's father, Barack Obama Sr. (1934–1982), was a married Luo Kenyan from Nyang'oma Kogelo. His last name, Obama, was derived from his Luo descent. Obama's parents met in 1960 in a Russian language class at the University of Hawaiʻi at Mānoa, where his father was a foreign student on a scho

## Challenge 5: Implement Retrieval Function

**Goal:** Create reusable document retrieval.

**Steps:**
1. Write function: retrieve_documents(query, k=3)
2. Query the vector database
3. Return top k most similar chunks
4. Include similarity scores
5. Test with 5 different queries

**Example queries:**
- "What is deep learning?"
- "How do transformers work?"
- "Explain neural networks"

**Expected:**
- Fast retrieval (< 1 second)
- Relevant results
- Proper ranking by similarity

In [11]:
def retrieve_documents(query, k=3):
    query_embedding = embedding_model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "rank": rank + 1,
            "distance": float(distances[0][rank]),
            "chunk_id": chunked_docs[idx]["chunk_id"],
            "text": chunked_docs[idx]["text"],
            "metadata": chunked_docs[idx]["metadata"]
        })
    return results

In [12]:
results = retrieve_documents("Who is Barack Obama?", k=3)

for res in results:
    print(f"\nRank: {res['rank']}")
    print(f"Distance: {res['distance']:.4f}")
    print(f"Chunk ID: {res['chunk_id']}")
    print(f"Title: {res['metadata']['title']}")
    print(res["text"])


Rank: 1
Distance: 0.3944
Chunk ID: 0
Title: Barack Obama
Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.

Rank: 2
Distance: 0.7740
Chunk ID: 10
Title: Barack Obama
African man who lived in the Colony of Virginia during the seventeenth century. Obama has described the ancestors of his grandparents as Scotch-Irish mostly. Obama's father, Barack Obama Sr. (1934–1982), was a married Luo Kenyan from Nyang'oma Kogelo. His last name, Obama, was derived from his Luo descent. Obama's parents met in 1960 in a Russian language class at the University of Hawaiʻi at Mānoa, where his father was a foreign student on a scholarship. The couple married in Wailuku, Hawaii, on February 2, 1961, six m

In [13]:
results = retrieve_documents("Where did Barack Obama go to college?", k=3)

for res in results:
    print(f"\nRank: {res['rank']}")
    print(f"Distance: {res['distance']:.4f}")
    print(f"Chunk ID: {res['chunk_id']}")
    print(f"Title: {res['metadata']['title']}")
    print(res["text"])


Rank: 1
Distance: 0.5221
Chunk ID: 20
Title: Barack Obama
lived off-campus on West 109th Street. He graduated with a Bachelor of Arts degree in 1983 and a 3.7 GPA. After graduating, Obama worked for about a year at the Business International Corporation, where he was a financial researcher and writer, then as a project coordinator for the New York Public Interest Research Group on the City College of New York campus for three months in 1985.

Rank: 2
Distance: 0.6149
Chunk ID: 11
Title: Barack Obama
In late August 1961, a few weeks after he was born, Barack and his mother moved to the University of Washington in Seattle, where they lived for a year. During that time, Barack's father completed his undergraduate degree in economics in Hawaii, graduating in June 1962. He left to attend graduate school on a scholarship at Harvard University, where he earned a Master of Arts in economics. Obama's parents divorced in March 1964. Obama Sr. returned to Kenya in 1964, where he married for a th

In [14]:
results = retrieve_documents("Who is Michelle Obama?", k=3)

for res in results:
    print(f"\nRank: {res['rank']}")
    print(f"Distance: {res['distance']:.4f}")
    print(f"Chunk ID: {res['chunk_id']}")
    print(f"Title: {res['metadata']['title']}")
    print(res["text"])


Rank: 1
Distance: 0.7892
Chunk ID: 32
Title: Barack Obama
In June 1989, Obama met Michelle Robinson when he was employed at Sidley Austin. Robinson was assigned for three months as Obama's adviser at the firm, and she joined him at several group social functions but declined his initial requests to date. They began dating later that summer, became engaged in 1991, and were married on October 3, 1992. After suffering a miscarriage, Michelle underwent in vitro fertilization to conceive their children. The couple's first daughter, Malia Ann, was born in 1998, followed by a second daughter, Natasha ("Sasha"), in 2001. The Obama daughters attended the University of Chicago Laboratory Schools. When they moved to Washington, D.C., in January 2009, the girls started at the Sidwell Friends School. The Obamas had two Portuguese Water Dogs; the first, a

Rank: 2
Distance: 0.9273
Chunk ID: 31
Title: Barack Obama
Obama lived with anthropologist Sheila Miyoshi Jager while he was a community organiz

## Challenge 6: Prompt Engineering

**Goal:** Write effective prompts for LLM context.

**Steps:**
1. Create system prompt defining bot role
2. Create instruction template for answering with context
3. Test different prompt variations
4. Evaluate which produces better answers

**Example prompts:**
```
System: "You are a helpful AI assistant for our documentation."

Template: 
"Based on the following documents:\n{context}\n\n
User question: {query}\n
Provide a helpful answer based only on the documents."
```

**Expected:**
- Clear, specific prompts
- Instructions prevent hallucination
- LLM answers based on retrived docs

In [15]:
def build_prompt(query, retrieved_docs):
    context = "\n\n".join([f"Source {doc['chunk_id']}:\n{doc['text']}" for doc in retrieved_docs])

    prompt = f"""You are a helpful assistant. Use the following retrieved documents to answer the question.
    if the answer is not in the documents, say "I don't have enough information to answer". Do not make up an answer.

    retrieved context:
    {context}

    user question:
    {query}
    """
    return prompt



In [16]:
retrieved_docs = retrieve_documents("Who is Barack Obama?", k=3)
prompt = build_prompt("Who is Barack Obama?", retrieved_docs)

print(prompt[:2000])

You are a helpful assistant. Use the following retrieved documents to answer the question.
    if the answer is not in the documents, say "I don't have enough information to answer". Do not make up an answer.

    retrieved context:
    Source 0:
Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.

Source 10:
African man who lived in the Colony of Virginia during the seventeenth century. Obama has described the ancestors of his grandparents as Scotch-Irish mostly. Obama's father, Barack Obama Sr. (1934–1982), was a married Luo Kenyan from Nyang'oma Kogelo. His last name, Obama, was derived from his Luo descent. Obama's parents met in 1960 in a Russian language class at the University o

## Challenge 7: Generate Answers with Context

**Goal:** Combine retrieval + LLM for informed answers.

**Steps:**
1. Create function: answer_with_context(query)
2. Retrieve relevant documents
3. Build context string from top 3 chunks
4. Call LLM with prompt + context
5. Return answer
6. Test on 5 queries

**Expected:**
- Answers grounded in retrieved documents
- No hallucinations
- Clear, helpful responses

In [17]:
def answer_with_context(query, k=3):
    retrieved_docs = retrieve_documents(query, k=k)
    prompt = build_prompt(query, retrieved_docs)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Use the following retrieved documents Only to answer the question. If the answer is not in the documents, say 'I don't have enough information to answer'. Do not make up an answer."},
            {"role": "user", "content": prompt}
        ]
    )

    answer=response.choices[0].message.content

    return{
        "query": query,
        "answer": answer,
        "sources": retrieved_docs
    }

In [18]:
result = answer_with_context("Who is Barack Obama?", k=3)

print("Answer:\n")
print(result["answer"])

print("\nSources used:\n")
for src in result["sources"]:
    print(f"Rank {src['rank']} | Distance: {src['distance']:.4f}")
    print(src["text"][:300])
    print("-" * 80)

Answer:

Barack Hussein Obama II is an American politician who served as the 44th president of the United States from 2009 to 2017. He is a member of the Democratic Party and was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004. He was born on August 4, 1961, in Honolulu, Hawaii.

Sources used:

Rank 1 | Distance: 0.3944
Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to
--------------------------------------------------------------------------------
Rank 2 | Distance: 0.7740
African man who lived in the Colony of Virginia during the seventeenth century. Obama has described the ancestors of his grandparents as Scotch-Iri

## Challenge 8: Implement Chat Memory

**Goal:** Allow multi-turn conversations.

**Steps:**
1. Create conversation history (list of messages)
2. Implement function: chat(user_message)
3. Maintain context from previous messages
4. Use conversation history in prompts
5. Test multi-turn conversation:
   - User: "Tell me about X"
   - Bot: Answer
   - User: "Can you elaborate?"
   - Bot: Uses conversation context

**Expected:**
- Bot remembers previous messages
- Answers are contextually appropriate
- Conversation flows naturally

In [19]:
conversation_history = []

def build_prompt_with_memory(query, retrieved_docs,conversation_history):
    context = "\n\n".join([f"Source {doc['chunk_id']}:\n{doc['text']}" for doc in retrieved_docs])

    prompt = f"""You are a helpful assistant. Use the following retrieved documents to answer the question.
    if the answer is not in the documents, say "I don't have enough information to answer". Do not make up an answer.

    conversation history:
    {conversation_history}

    retrieved context:
    {context}

    user question:
    {query}
    """
    return prompt


In [20]:
def chat(user_message, k=3):
    global conversation_history
    retrieved_docs = retrieve_documents(user_message, k=k)
    prompt = build_prompt_with_memory(user_message, retrieved_docs,conversation_history)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Use the following retrieved documents Only to answer the question. If the answer is not in the documents, say 'I don't have enough information to answer'. Do not make up an answer."},
            {"role": "user", "content": prompt}
        ]
    )

    answer=response.choices[0].message.content

    conversation_history.append({"role": "user", "content": user_message})
    conversation_history.append({"role": "assistant", "content": answer})
    
    return{
        "answer": answer,
        "sources": retrieved_docs
    }

In [21]:
user_questions = [
    "Who is Barack Obama?",
    "Where was he born?",
    "What political party was he in?",
    "Who is his wife?"
]
for question in user_questions:
    print(f"\nUser: {question}")
    response = chat(question)
    print("Bot:", response["answer"])


User: Who is Barack Obama?
Bot: Barack Hussein Obama II is an American politician who served as the 44th president of the United States from 2009 to 2017. He was a member of the Democratic Party and was the first African American president. Before his presidency, he served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004. He was born on August 4, 1961, in Honolulu, Hawaii.

User: Where was he born?
Bot: Barack Obama was born in Honolulu, Hawaii.

User: What political party was he in?
Bot: Barack Obama was a member of the Democratic Party.

User: Who is his wife?
Bot: I don't have enough information to answer.


## Challenge 9: Error Handling and Fallbacks

**Goal:** Make bot robust to failures.

**Steps:**
1. Handle API errors (rate limits, connection errors)
2. Handle no relevant documents found
3. Implement fallback responses
4. Add logging for debugging
5. Test edge cases:
   - Query that's not in documents
   - Empty query
   - Very long query

**Expected:**
- Graceful error messages
- Bot doesn't crash
- Helpful fallbacks

In [22]:
def robust_chat(user_message, k=3):
    global conversation_history

    if not user_message or not user_message.strip():
        return {
            "answer": "Please enter a valid question.",
            "sources": []
        }

    try:
        retrieved_docs = retrieve_documents(user_message, k=k)

        if not retrieved_docs:
            return {
                "answer": "Unable to retrieve relevant documents for your question.",
                "sources": []
            }
        
        prompt = build_prompt_with_memory(user_message, retrieved_docs,conversation_history)

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Use the following retrieved documents Only to answer the question. If the answer is not in the documents, say 'I don't have enough information to answer'. Do not make up an answer."},
                {"role": "user", "content": prompt}
            ]
        )

        answer=response.choices[0].message.content

        conversation_history.append({"role": "user", "content": user_message})
        conversation_history.append({"role": "assistant", "content": answer})
        
        return{
            "answer": answer,
            "sources": retrieved_docs
        }
    except Exception as e:
        return {
            "answer": "An error occurred while processing your request.",
            "error": str(e),
            "sources": []
        }

In [23]:
test_questions = [
    "",
    "   ",
    "Where is Obama from?",
    "is tom brady the greatest quarterback of all time?"
]
for question in test_questions:
    print(f"\nUser: '{question}'")
    response = robust_chat(question)
    print("Bot:", response["answer"])


User: ''
Bot: Please enter a valid question.

User: '   '
Bot: Please enter a valid question.

User: 'Where is Obama from?'
Bot: I don't have enough information to answer.

User: 'is tom brady the greatest quarterback of all time?'
Bot: I don't have enough information to answer.


## Challenge 10: Performance Evaluation

**Goal:** Measure retrieval and answer quality.

**Steps:**
1. Create test set of questions (10+)
2. For each question:
   - Query the bot
   - Manually rate answer (1-5 scale)
   - Check if answer uses correct documents
   - Measure response time
3. Calculate metrics:
   - Average rating
   - Accuracy: % correct documents retrieved
   - Speed: average response time

**Expected:**
- Average rating > 3.5
- Retrieval accuracy > 80%
- Response time < 2 seconds

In [24]:
test_questions = [
    "Who is Barack Obama?",
    "When was Barack Obama born?",
    "Where was Barack Obama born?",
    "What political party does Barack Obama belong to?",
    "What office did Barack Obama hold?",
    "What did Obama do before becoming president?",
    "Who was Obama's father?",
    "Where did Obama's parents meet?",
    "Was Obama the first African American president?",
    "Who is Obama's wife?"
]

In [32]:
import time
import pandas as pd

evaluation_results = []

for question in test_questions:
    start_time = time.time()
    response = robust_chat(question)
    end_time = time.time()

    evaluation_results.append({
        "question": question,
        "answer": response["answer"],
        "retrieved_chunk_ids": [doc["chunk_id"] for doc in response.get("sources", [])],
        "error": response.get("error", ""),
        "response_time_sec": end_time - start_time,
        "rating": None,  # Placeholder for manual rating: 1-5
        "retrieval_correct": None  # Placeholder for manual evaluation of retrieval relevance 1or 0
    })

df_eval = pd.DataFrame(evaluation_results)
df_eval

,question,answer,retrieved_chunk_ids,error,response_time_sec,rating,retrieval_correct
0,Who is Barack Obama?,Barack Hussein Obama II is an American politic...,"[0, 10, 9]",,3.878555,None,None
1,When was Barack Obama born?,"Barack Obama was born on August 4, 1961.","[0, 9, 10]",,0.973067,None,None
2,Where was Barack Obama born?,"Barack Obama was born in Honolulu, Hawaii.","[9, 10, 0]",,0.610571,None,None
3,What political party does Barack Obama belong to?,Barack Obama was a member of the Democratic Pa...,"[0, 2, 1]",,0.884736,None,None
4,What office did Barack Obama hold?,Barack Obama served as the 44th president of t...,"[0, 1, 180]",,0.692658,None,None
5,What did Obama do before becoming president?,"Before becoming president, Barack Obama served...","[0, 150, 65]",,1.089979,None,None
6,Who was Obama's father?,Barack Obama's father was Barack Obama Sr. He ...,"[10, 11, 9]",,0.810954,None,None
7,Where did Obama's parents meet?,Obama's parents met in a Russian language clas...,"[10, 31, 27]",,0.721616,None,None
8,Was Obama the first African American president?,"Yes, Barack Obama was the first African Americ...","[0, 185, 75]",,0.994420,None,None
9,Who is Obama's wife?,"Obama's wife is Michelle Robinson, whom he mar...","[31, 10, 32]",,1.103797,None,None


In [35]:
for i, row in df_eval.iterrows():
    print(f"\nQuestion {i+1}: {row['question']}")
    print(f"Answer: {row['answer']}")


Question 1: Who is Barack Obama?
Answer: Barack Hussein Obama II is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004. He was born on August 4, 1961, in Honolulu, Hawaii.

Question 2: When was Barack Obama born?
Answer: Barack Obama was born on August 4, 1961.

Question 3: Where was Barack Obama born?
Answer: Barack Obama was born in Honolulu, Hawaii.

Question 4: What political party does Barack Obama belong to?
Answer: Barack Obama was a member of the Democratic Party.

Question 5: What office did Barack Obama hold?
Answer: Barack Obama served as the 44th president of the United States from 2009 to 2017.

Question 6: What did Obama do before becoming president?
Answer: Before becoming president, Barack Obama served as a U.S. sena

In [38]:
df_eval["notes"] = [
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",    
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context.",
    "Accurate answer, directly from retrieved context."
]

In [39]:
df_eval["rating"] = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
df_eval["retrieval_correct"] = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

df_eval

,question,answer,retrieved_chunk_ids,error,response_time_sec,rating,retrieval_correct,notes
0,Who is Barack Obama?,Barack Hussein Obama II is an American politic...,"[0, 10, 9]",,3.878555,5,1,"Accurate answer, directly from retrieved context."
1,When was Barack Obama born?,"Barack Obama was born on August 4, 1961.","[0, 9, 10]",,0.973067,5,1,"Accurate answer, directly from retrieved context."
2,Where was Barack Obama born?,"Barack Obama was born in Honolulu, Hawaii.","[9, 10, 0]",,0.610571,5,1,"Accurate answer, directly from retrieved context."
3,What political party does Barack Obama belong to?,Barack Obama was a member of the Democratic Pa...,"[0, 2, 1]",,0.884736,5,1,"Accurate answer, directly from retrieved context."
4,What office did Barack Obama hold?,Barack Obama served as the 44th president of t...,"[0, 1, 180]",,0.692658,5,1,"Accurate answer, directly from retrieved context."
5,What did Obama do before becoming president?,"Before becoming president, Barack Obama served...","[0, 150, 65]",,1.089979,5,1,"Accurate answer, directly from retrieved context."
6,Who was Obama's father?,Barack Obama's father was Barack Obama Sr. He ...,"[10, 11, 9]",,0.810954,5,1,"Accurate answer, directly from retrieved context."
7,Where did Obama's parents meet?,Obama's parents met in a Russian language clas...,"[10, 31, 27]",,0.721616,5,1,"Accurate answer, directly from retrieved context."
8,Was Obama the first African American president?,"Yes, Barack Obama was the first African Americ...","[0, 185, 75]",,0.994420,5,1,"Accurate answer, directly from retrieved context."
9,Who is Obama's wife?,"Obama's wife is Michelle Robinson, whom he mar...","[31, 10, 32]",,1.103797,5,1,"Accurate answer, directly from retrieved context."


In [37]:
average_rating = df_eval["rating"].mean()
retrieval_accuracy = df_eval["retrieval_correct"].mean() * 100
average_response_time = df_eval["response_time_sec"].mean()

print("Average rating:", round(average_rating, 2))
print("Retrieval accuracy:", round(retrieval_accuracy, 2), "%")
print("Average response time:", round(average_response_time, 3), "seconds")

Average rating: 5.0
Retrieval accuracy: 100.0 %
Average response time: 1.176 seconds


# RAG System Performance Report

## Overview

In this project, I built a Retrieval-Augmented Generation (RAG) system that retrieves relevant document chunks using vector embeddings and generates answers using a language model. The goal was to ensure that responses are grounded in retrieved context rather than relying on unsupported or hallucinated information.

## Evaluation Results

- Average Rating: 5.0
- Retrieval Accuracy: 100%
- Average Response Time: 1.17 seconds

## Interpretation of Results

The system performed very well on the evaluation set. All answers were accurate, directly addressed the questions, and were supported by the retrieved document chunks. The retrieval component consistently returned relevant context, allowing the model to generate correct and grounded responses.

The response time remained under two seconds on average, indicating that the system is efficient and suitable for real-time use.

## Strengths

- The system produces clear and accurate answers for factual questions.
- Retrieval is highly reliable, consistently returning relevant chunks.
- Responses remain grounded in the provided documents, avoiding hallucination.
- The overall pipeline is fast and efficient.

## Limitations

The evaluation was conducted using a single, well-structured Wikipedia document with relatively straightforward questions. Because of this, the system has not yet been tested under more complex or realistic conditions, such as:

- Multiple documents with overlapping or conflicting information
- Ambiguous or multi-step questions
- Noisy or unstructured data
- Queries that require reasoning across multiple sources

## Potential Improvents

- Test with multiple documents to evaluate cross-document retrieval
- Introduce more complex and open-ended questions
- Implement hybrid retrieval (combining keyword and vector search)
- Add reranking to improve the quality of retrieved results
- Expand evaluation to include harder and edge-case scenarios

## Conclusion

This project demonstrates a fully functional RAG pipeline, including document processing, embedding, retrieval, prompt design, and evaluation. While the system performs very well in a controlled setting, further improvements are needed to ensure robustness in more complex and realistic environments.

## Challenge 11: Web Interface (Optional)

**Goal:** Build simple UI for bot.

**Steps:**
1. Create Streamlit or FastAPI app
2. Input: text box for user query
3. Output: bot response + source documents
4. Display retrieval scores
5. Show conversation history

**To run:**
```bash
streamlit run app.py
# or
uvicorn app:app --reload
```

**Expected:**
- Interface loads without errors
- Bot responds to queries
- Sources displayed

In [26]:
# Challenge 11: Web Interface (Optional)

# YOUR CODE HERE

# This is best done in a separate file, but you can prototype here

## Challenge 12: Project Summary and Deployment (optional)

**Goal:** Document and deploy the bot.

**Steps:**
1. Write comprehensive README with:
   - Architecture overview
   - How to set up environment
   - How to run the bot
   - Example queries and responses
   - Limitations and improvements

2. Create deployment guide:
   - How to deploy to cloud (AWS/GCP/Azure)
   - Scaling considerations
   - Cost estimation

3. Include:
   - Architecture diagram (description)
   - Performance metrics
   - User feedback gathering

**Example structure:**
```markdown
# RAG Bot README

## Architecture
Documents → Split → Embed → Vector DB → Retrieval → LLM → Answer

## Performance
- Average retrieval accuracy: 85%
- Response time: 1.2 seconds
- User satisfaction: 4.2/5.0

## Deployment
- AWS Lambda for API
- DynamoDB for conversations
- CloudWatch for monitoring
```

In [27]:
# Challenge 12: Project Summary and Deployment

# YOUR CODE HERE